# NYUMets: what contrasts does each patient have?

Discovery-first EDA over `data/imaging/.../<patientId>/`. Nothing here assumes the NYUMets
naming convention -- the early cells **report the filename/directory vocabulary they find**,
and the contrast classifier is a small ordered rule table you edit once you have seen it.

Order of business:

0. locate the root, probe the directory layout
1. walk every patient -> file manifest (cached to CSV)
2. token census: what words actually appear in paths / filenames
3. classify files -> contrast labels; **report what did not match**
4. infer visit / session keys (longitudinal structure)
5. availability tables + plots: patient x contrast, session x contrast, co-occurrence, combos
6. header-only scan: shapes, voxel spacing, orientation, within-session affine agreement
7. per-contrast intensity histograms across subjects
8. eyeball one session
9. write the tidy `(patient, session, contrast)` CSV

Steps 0-5 are stdlib + numpy + matplotlib only (no pandas), so they run in a bare env. From step 6
on it also needs `nibabel` and the repo's own `visualization` helpers (`subplot_hists`,
`subplot_images`), which pull in torch.


## 0. Root + layout probe

In [ ]:
import os, re, sys, csv, json, time
from collections import Counter, defaultdict

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

# repo root (works whether the kernel cwd is ImMAP/ or ImMAP/notebooks/)
REPO = os.getcwd()
if os.path.basename(REPO) == "notebooks":
    REPO = os.path.dirname(REPO)
if REPO not in sys.path:
    sys.path.insert(0, REPO)

# the symlink lives beside the repo; the gpfs path is the fallback
CANDIDATES = [
    os.path.join(REPO, "..", "datasets", "NYUMets"),
    "/gpfs/data/fenglab/LiFeng/NYUMets/NYUMets",
]
ROOT = next((os.path.abspath(p) for p in CANDIDATES if os.path.isdir(p)), None)
assert ROOT is not None, "none of these exist:\n  " + "\n  ".join(map(os.path.abspath, CANDIDATES))

CACHE = os.path.join(REPO, "cache")
os.makedirs(CACHE, exist_ok=True)
MANIFEST_CSV = os.path.join(CACHE, "nyumets_manifest.csv")
TIDY_CSV     = os.path.join(CACHE, "nyumets_contrasts.csv")

# knobs
MAX_PATIENTS = None    # None = all; set e.g. 50 for a fast first pass over a slow FS
COLLECT_SIZE = True    # st_size per file: one extra stat each, but catches empty/truncated volumes

print("ROOT :", ROOT)
print("real :", os.path.realpath(ROOT))
print("cache:", CACHE)

In [ ]:
def tree(root, max_depth=3, max_entries=8, _depth=0, _prefix=""):
    """Depth- and width-limited directory listing. Directories first, files after."""
    try:
        entries = sorted(os.scandir(root), key=lambda e: (not e.is_dir(), e.name))
    except OSError as err:
        print(_prefix + "!! " + str(err))
        return
    shown = entries[:max_entries]
    for e in shown:
        print(_prefix + ("[d] " if e.is_dir() else "    ") + e.name)
        if e.is_dir() and _depth + 1 < max_depth:
            tree(e.path, max_depth, max_entries, _depth + 1, _prefix + "    ")
    if len(entries) > max_entries:
        print(_prefix + f"... (+{len(entries) - max_entries} more)")


print(f"=== {ROOT} ===")
tree(ROOT, max_depth=2, max_entries=12)

### Where do the patient folders start?

The stated layout is `data/imaging/patientId/<ID>/...`. We try that first and fall back through
the plausible parents. Whatever it picks is printed -- **check it before moving on** and hard-set
`PATIENT_ROOT` if the guess is wrong.

In [ ]:
def pick_patient_root(root):
    """First candidate that exists and holds many subdirectories -> the per-patient level."""
    rels = ["data/imaging/patientId", "data/imaging", "data", "imaging", ""]
    best = None
    for rel in rels:
        p = os.path.join(root, *rel.split("/")) if rel else root
        if not os.path.isdir(p):
            continue
        subs = [e.name for e in os.scandir(p) if e.is_dir()]
        print(f"{rel or '.':<24} {len(subs):>6} subdirs   e.g. {sorted(subs)[:4]}")
        if best is None and len(subs) >= 10:
            best = p
    return best


PATIENT_ROOT = pick_patient_root(ROOT)
assert PATIENT_ROOT, "no directory level looked like a patient list -- set PATIENT_ROOT by hand"
print("\nPATIENT_ROOT =", PATIENT_ROOT)

PATIENTS = sorted(e.name for e in os.scandir(PATIENT_ROOT) if e.is_dir())
if MAX_PATIENTS:
    PATIENTS = PATIENTS[:MAX_PATIENTS]
print(f"{len(PATIENTS)} patients   first: {PATIENTS[:5]}   last: {PATIENTS[-3:]}")

In [ ]:
# what does the inside of a patient folder look like?
for pid in PATIENTS[:3]:
    print(f"=== {pid} ===")
    tree(os.path.join(PATIENT_ROOT, pid), max_depth=3, max_entries=10)
    print()

## 1. Walk every patient -> manifest

One row per file, `.nii/.nii.gz` and everything else alike -- the non-image files (JSON sidecars,
CSV metadata, DICOM leftovers) tell you as much about the layout as the volumes do.
Cached to CSV; set `REWALK=True` to redo it.

In [ ]:
REWALK = False

NII_RE = re.compile(r"\.nii(\.gz)?$", re.I)


def ext_of(fname):
    """'.nii.gz' kept whole; otherwise the last suffix. '' for extensionless files."""
    low = fname.lower()
    if low.endswith(".nii.gz"):
        return ".nii.gz"
    return os.path.splitext(low)[1]


def stem_of(fname):
    e = ext_of(fname)
    return fname[: -len(e)] if e else fname


def walk_patients(patient_root, patients, collect_size=True):
    rows = []
    t0 = time.time()
    for i, pid in enumerate(patients):
        pdir = os.path.join(patient_root, pid)
        for dirpath, _dirnames, filenames in os.walk(pdir):
            rel = os.path.relpath(dirpath, pdir)
            rel = "" if rel == "." else rel.replace(os.sep, "/")
            for fn in filenames:
                nbytes = -1
                if collect_size:
                    try:
                        nbytes = os.stat(os.path.join(dirpath, fn)).st_size
                    except OSError:
                        pass
                rows.append({"patient": pid, "reldir": rel, "fname": fn,
                             "ext": ext_of(fn), "nbytes": nbytes})
        if (i + 1) % 100 == 0:
            print(f"  {i+1}/{len(patients)} patients, {len(rows)} files, {time.time()-t0:.0f}s")
    return rows


FIELDS = ["patient", "reldir", "fname", "ext", "nbytes"]

if REWALK or not os.path.exists(MANIFEST_CSV):
    rows = walk_patients(PATIENT_ROOT, PATIENTS, COLLECT_SIZE)
    with open(MANIFEST_CSV, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=FIELDS)
        w.writeheader()
        w.writerows(rows)
    print("wrote", MANIFEST_CSV)
else:
    with open(MANIFEST_CSV, newline="") as f:
        rows = list(csv.DictReader(f))
    for r in rows:
        r["nbytes"] = int(r["nbytes"])
    print("loaded cache", MANIFEST_CSV)

print(f"{len(rows)} files across {len({r['patient'] for r in rows})} patients")

In [ ]:
ext_counts = Counter(r["ext"] for r in rows)
print("file extensions:")
for e, n in ext_counts.most_common(20):
    tot = sum(r["nbytes"] for r in rows if r["ext"] == e and r["nbytes"] > 0)
    print(f"  {e or '(none)':<12} {n:>8}   {tot/2**30:>8.2f} GiB")

nii = [r for r in rows if NII_RE.search(r["fname"])]
print(f"\n{len(nii)} NIfTI files")

empty = [r for r in nii if 0 <= r["nbytes"] < 1024]
print(f"{len(empty)} suspiciously small (<1 KiB) NIfTIs"
      + (f", e.g. {[r['fname'] for r in empty[:3]]}" if empty else ""))

depth = Counter(r["reldir"].count("/") + (1 if r["reldir"] else 0) for r in nii)
print("\ndirectory depth below the patient folder (NIfTIs only):")
for d, n in sorted(depth.items()):
    print(f"  depth {d}: {n}")

## 2. Token census -- the naming vocabulary

This is the cell that actually drives the contrast rules below. It splits every intermediate
directory name and every filename stem into alphanumeric tokens and counts them. Contrast tags,
sequence names, visit markers and derived-file suffixes all fall out here.

In [ ]:
TOKEN_RE = re.compile(r"[^a-z0-9]+")

def toks(s):
    return [t for t in TOKEN_RE.split(str(s).lower()) if t]

def mask(s):
    """Collapse digit runs: '20190505' -> '#', 'NYU0042' -> 'NYU#'. Dates and patient IDs
    otherwise swamp every count and hide the handful of tokens that carry meaning."""
    return re.sub(r"\d+", "#", str(s))


dir_parts  = [p for r in nii for p in r["reldir"].split("/") if p]
file_stems = [stem_of(r["fname"]) for r in nii]

def show(counter, title, n=40):
    print(f"--- {title} ({len(counter)} distinct) ---")
    if not counter:
        print("  (none)")
    for t, c in counter.most_common(n):
        print(f"  {t:<24} {c}")
    print()

show(Counter(t for p in dir_parts  for t in toks(mask(p))), "directory-name tokens (masked)")
show(Counter(t for s in file_stems for t in toks(mask(s))), "filename tokens (masked)")

# short bare numbers survive masking as a category of their own -- echo index, series number,
# b-value. Long ones are dates/IDs and are already covered by the masked counts above.
show(Counter(t for s in file_stems for t in toks(s) if t.isdigit() and len(t) <= 4),
     "short numeric filename tokens (<=4 digits, unmasked)", n=20)

In [ ]:
# full directory names and full filenames -- tokens alone can hide the real convention
print("--- distinct reldir shapes, digit runs masked (top 25) ---")
for d, c in Counter(mask(r["reldir"]) for r in nii).most_common(25):
    print(f"  {c:>7}  {d or '(patient root)'}")

print("\n--- distinct filenames, digit runs masked (top 40) ---")
for f_, c in Counter(mask(r["fname"]) for r in nii).most_common(40):
    print(f"  {c:>7}  {f_}")

## 3. Classify files -> contrast labels

Ordered rules, **first match wins**, applied to `reldir + "/" + filename` lowercased. Order is
load-bearing: `seg` before everything (a `*_t1c_seg` is a label, not an image), `FLAIR` before
`T2` (`t2_flair` is FLAIR), `T1ce` before `T1`, `ADC` before `DWI`.

Every rule that matches is recorded, not just the winner, so rule collisions show up as
`ambiguous` rather than silently resolving. **Edit this table after reading section 2**, then
re-run from here -- nothing above depends on it.

In [ ]:
CONTRAST_RULES = [
    # (label, regex over the lowercased "reldir/filename")
    ("seg",   r"(seg|label|lesion|mask|tumou?r|gtv|_roi|annotation|contour)"),
    ("ADC",   r"(adc|apparent[\W_]?diff)"),
    ("DWI",   r"(dwi|dti|trace|b1000|diffusion)"),
    ("SWI",   r"(swi|gre\b|t2star|t2\*|susceptib|medic)"),
    ("FLAIR", r"(flair|dark[\W_]?fluid)"),
    # NYUMets writes the contrast-enhanced T1 as CT1 -- the modifier PRECEDES the t1, which the
    # usual "t1 + suffix" patterns all miss. Without the leading-c alternative these files fall
    # through to the T1 rule (t1 is a substring of ct1) and silently pollute the T1 bucket.
    # The guard must be a lookbehind, NOT \bct1\b: '_' is a word character, so \b never fires
    # between the '_' and the 'c' in 'NYU0001_CT1.nii.gz'. (?!\d) keeps CT10 out.
    ("T1ce",  r"((?<![a-z0-9])c[\W_]?t1(?!\d)|"
              r"t1[\W_]*(ce|c\b|post|gd|gad|contrast)|post[\W_]?contrast|"
              r"(mprage|bravo|spgr|tfe)[\W_]*(post|gd|c\b)|t1[\W_]?w?[\W_]?gd)"),
    ("T1",    r"(t1|mprage|bravo|spgr|tfe|mp[\W_]?rage)"),
    ("T2",    r"(t2|tse|cube|space)"),
]
CONTRAST_RES = [(lab, re.compile(pat, re.I)) for lab, pat in CONTRAST_RULES]
LABELS = [lab for lab, _ in CONTRAST_RULES]


def classify(reldir, fname):
    """-> (winning label or None, tuple of every label whose rule matched)."""
    s = (reldir + "/" + fname).lower()
    hits = tuple(lab for lab, rx in CONTRAST_RES if rx.search(s))
    return (hits[0] if hits else None), hits


for r in nii:
    r["contrast"], r["hits"] = classify(r["reldir"], r["fname"])

print("classified:")
for lab, n in Counter(r["contrast"] for r in nii).most_common():
    print(f"  {str(lab):<8} {n:>7}")

amb = [r for r in nii if len(r["hits"]) > 1]
print(f"\n{len(amb)} files matched >1 rule (winner = first). Collision patterns:")
for combo, n in Counter(r["hits"] for r in amb).most_common(15):
    ex = next(r["fname"] for r in amb if r["hits"] == combo)
    print(f"  {n:>7}  {' > '.join(combo):<28} e.g. {ex}")

In [ ]:
# --- the diagnostic that matters: what did NOT match any rule ---
un = [r for r in nii if r["contrast"] is None]
print(f"{len(un)} unclassified NIfTIs ({100*len(un)/max(len(nii),1):.1f}%)\n")

print("--- unclassified filenames, digit runs masked (top 40) ---")
for f_, c in Counter(mask(r["fname"]) for r in un).most_common(40):
    print(f"  {c:>7}  {f_}")

print("\n--- tokens in unclassified files, masked (top 30) ---")
for t, c in Counter(t for r in un for t in toks(mask(stem_of(r["fname"])))).most_common(30):
    print(f"  {t:<22} {c}")

print("\n--- and their parent directories, masked (top 15) ---")
for d, c in Counter(mask(r["reldir"]) for r in un).most_common(15):
    print(f"  {c:>7}  {d or '(patient root)'}")

## 4. Visit / session keys

NYUMets is longitudinal, so "which contrasts does patient X have" is really
"which contrasts does patient X have *at visit V*". We look for a session key in this order:
an ISO-ish or 8-digit date anywhere in the path, then a `ses/visit/scan/study/tp` marker, then
the first directory component below the patient folder, then `(single)` for a flat layout.

If the keys come out as dates they sort chronologically and visit ordinals are meaningful;
if not, the ordinal is just a stable arbitrary index. The cell says which case you are in.

In [ ]:
DATE_RE = re.compile(r"((?:19|20)\d{2})[-_.]?(\d{2})[-_.]?(\d{2})")
SESS_RE = re.compile(r"\b(?:ses|session|visit|scan|study|timepoint|tp)[-_ ]?(\d+)\b", re.I)


def session_key(reldir, fname):
    for src in (reldir, fname):
        m = DATE_RE.search(src)
        if m:
            return "-".join(m.groups()), "date"
    for src in (reldir, fname):
        m = SESS_RE.search(src)
        if m:
            return "ses%03d" % int(m.group(1)), "marker"
    if reldir:
        # The IMMEDIATE PARENT directory, not the first component. NYUMets nests as
        # <patient>/studyId/<STUDY_ID>/FLAIR.nii, where `studyId` is a fixed literal folder:
        # taking the first component returns "studyId" for every file, collapsing all of a
        # patient's studies into one pseudo-session whose contrasts then come from DIFFERENT
        # dates. The last component is the folder the contrasts were acquired into, which is
        # the grouping that actually means "one session" in every layout seen so far.
        return reldir.split("/")[-1], "dirname"
    return "(single)", "flat"


for r in nii:
    r["session"], r["session_src"] = session_key(r["reldir"], r["fname"])

print("session key source:", dict(Counter(r["session_src"] for r in nii)))

sess_by_pat = defaultdict(set)
for r in nii:
    sess_by_pat[r["patient"]].add(r["session"])

counts = [len(v) for v in sess_by_pat.values()]
nsess = Counter(counts)
print(f"\nsessions per patient: min {min(counts)}, max {max(counts)}, mean {np.mean(counts):.2f}")
for k in sorted(nsess)[:15]:
    print(f"  {k:>3} session(s): {nsess[k]:>5} patients")

ex = max(sess_by_pat, key=lambda p: len(sess_by_pat[p]))
print(f"\nmost visits: {ex} -> {sorted(sess_by_pat[ex])[:12]}")

## 5. Availability

In [ ]:
# (patient, session) -> {contrast: [files]}
cell = defaultdict(lambda: defaultdict(list))
for r in nii:
    if r["contrast"]:
        cell[(r["patient"], r["session"])][r["contrast"]].append(r)

pat_contrasts = defaultdict(set)          # pooled over all visits
for (p, s), d in cell.items():
    pat_contrasts[p] |= set(d)

n_pat  = len(PATIENTS)
n_sess = len(cell)
print(f"{n_pat} patients, {n_sess} (patient, session) cells\n")

print(f"{'contrast':<8} {'patients':>10} {'%':>6}   {'sessions':>10} {'%':>6}")
for lab in LABELS:
    np_ = sum(lab in v for v in pat_contrasts.values())
    ns_ = sum(lab in d for d in cell.values())
    print(f"{lab:<8} {np_:>10} {100*np_/max(n_pat,1):>5.1f}%   "
          f"{ns_:>10} {100*ns_/max(n_sess,1):>5.1f}%")

dupes = [(k, lab, len(v)) for k, d in cell.items() for lab, v in d.items() if len(v) > 1]
print(f"\n{len(dupes)} (patient, session, contrast) cells hold >1 file "
      "(repeat acquisitions, echoes, or an over-broad rule)")
for k, lab, n in dupes[:8]:
    print(f"  {k} {lab} x{n}: {[r['fname'] for r in cell[k][lab]][:3]}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
xs = np.arange(len(LABELS))

axes[0].bar(xs, [sum(l in v for v in pat_contrasts.values()) for l in LABELS], color="#4878a8")
axes[0].set_title("patients with >=1 volume of each contrast")
axes[1].bar(xs, [sum(l in d for d in cell.values()) for l in LABELS], color="#a85448")
axes[1].set_title("(patient, session) cells with each contrast")
for ax in axes:
    ax.set_xticks(xs); ax.set_xticklabels(LABELS, rotation=45, ha="right")
    ax.grid(axis="y", alpha=.3)
plt.tight_layout(); plt.show()

In [ ]:
# binary availability heatmap, patients grouped by which set they have
order = sorted(pat_contrasts, key=lambda p: (tuple(l in pat_contrasts[p] for l in LABELS), p),
               reverse=True)
A = np.array([[l in pat_contrasts[p] for l in LABELS] for p in order], dtype=float)

fig, ax = plt.subplots(figsize=(0.55 * len(LABELS) + 3, 7))
ax.imshow(A, aspect="auto", cmap="Greys", interpolation="nearest", vmin=0, vmax=1)
ax.set_xticks(range(len(LABELS))); ax.set_xticklabels(LABELS, rotation=45, ha="right")
ax.set_ylabel(f"patients (n={len(order)}, sorted by contrast set)")
ax.set_title("contrast availability per patient (pooled over visits)")
plt.tight_layout(); plt.show()

In [ ]:
# which contrast SETS occur, at the session level
combos = Counter(frozenset(d) for d in cell.values())
print(f"{len(combos)} distinct contrast sets over {n_sess} sessions\n")
print(f"{'n':>7}  {'%':>6}  set")
for cset, n in combos.most_common(20):
    print(f"{n:>7}  {100*n/n_sess:>5.1f}%  {sorted(cset, key=LABELS.index)}")

In [ ]:
# co-occurrence at the session level: P(col | row)
M = np.zeros((len(LABELS), len(LABELS)))
for d in cell.values():
    present = [i for i, l in enumerate(LABELS) if l in d]
    for i in present:
        for j in present:
            M[i, j] += 1
P = M / np.maximum(np.diag(M), 1)[:, None]

fig, ax = plt.subplots(figsize=(6.5, 5.5))
im = ax.imshow(P, cmap="viridis", vmin=0, vmax=1)
ax.set_xticks(range(len(LABELS))); ax.set_xticklabels(LABELS, rotation=45, ha="right")
ax.set_yticks(range(len(LABELS))); ax.set_yticklabels(LABELS)
for i in range(len(LABELS)):
    for j in range(len(LABELS)):
        ax.text(j, i, f"{P[i,j]:.2f}", ha="center", va="center",
                color="w" if P[i, j] < .6 else "k", fontsize=8)
ax.set_title("P(column present | row present), per session")
fig.colorbar(im, ax=ax, fraction=.046); plt.tight_layout(); plt.show()

### The query that actually gates the synthesis work

How much of this dataset is usable as BraTS-style multi-contrast input, i.e. how many
**visits** carry the full quartet at once (and how many patients have at least one such visit).
Change `CORE` to test other requirements -- e.g. `("T1", "T1ce")` for a bare T1 -> T1ce bridge.

In [ ]:
CORE = ("FLAIR", "T1", "T1ce", "T2")

full_sess = [k for k, d in cell.items() if all(l in d for l in CORE)]
full_pats = {p for p, _ in full_sess}
print(f"CORE = {CORE}")
print(f"  {len(full_sess)}/{n_sess} sessions ({100*len(full_sess)/max(n_sess,1):.1f}%) complete")
print(f"  {len(full_pats)}/{n_pat} patients ({100*len(full_pats)/max(n_pat,1):.1f}%) "
      "have >=1 complete visit")

per_pat = Counter(p for p, _ in full_sess)
print(f"\ncomplete visits per patient (of those that have any): "
      f"max {max(per_pat.values()) if per_pat else 0}")
for k, v in sorted(Counter(per_pat.values()).items())[:12]:
    print(f"  {k:>3} complete visit(s): {v:>5} patients")

print("\nmissing-contrast breakdown over incomplete sessions:")
inc = [d for k, d in cell.items() if not all(l in d for l in CORE)]
for lab, c in Counter(l for d in inc for l in CORE if l not in d).most_common():
    print(f"  missing {lab:<6} {c:>7}  ({100*c/max(len(inc),1):.1f}% of incomplete)")

## 6. Header-only scan: geometry

`nib.load` is lazy -- the header comes back without touching the voxel data, so this is cheap
even over gpfs. Two things we need to know before any of this is trainable:

* is every volume on a **common grid** (already resampled / skull-stripped like BraTS), or raw
  scanner geometry with per-sequence spacing?
* within one visit, do the contrasts share an affine, i.e. are they **co-registered**?

In [ ]:
import nibabel as nib

HDR_SAMPLE = 150        # files per contrast

def header_info(path):
    img = nib.load(path)                     # lazy: no voxel data read
    hdr = img.header
    return {"shape": tuple(int(s) for s in img.shape[:3]),
            "zooms": tuple(round(float(z), 2) for z in hdr.get_zooms()[:3]),
            "axcodes": "".join(nib.aff2axcodes(img.affine)),
            "dtype": str(hdr.get_data_dtype())}


def full_path(r):
    return os.path.join(PATIENT_ROOT, r["patient"], r["reldir"], r["fname"])


rng = np.random.default_rng(0)
for lab in LABELS:
    pool = [r for r in nii if r["contrast"] == lab]
    if not pool:
        continue
    sel = [pool[i] for i in rng.permutation(len(pool))[:HDR_SAMPLE]]
    infos, bad = [], 0
    for r in sel:
        try:
            infos.append(header_info(full_path(r)))
        except Exception:
            bad += 1
    if not infos:
        print(f"{lab}: {bad} read failures\n"); continue
    print(f"--- {lab}  (n={len(infos)} sampled of {len(pool)}"
          + (f", {bad} unreadable" if bad else "") + ") ---")
    for key in ("shape", "zooms", "axcodes", "dtype"):
        top = Counter(i[key] for i in infos).most_common(4)
        print(f"  {key:<8} " + "  |  ".join(f"{v} x{c}" for v, c in top))
    print()

In [ ]:
# co-registration check: within a session, do contrasts share shape + affine?
SESS_SAMPLE = 60

keys = [k for k, d in cell.items() if len(d) >= 2]
sel = [keys[i] for i in rng.permutation(len(keys))[:SESS_SAMPLE]]

same_shape = same_affine = total = 0
for k in sel:
    firsts = [v[0] for v in cell[k].values()]
    try:
        imgs = [nib.load(full_path(r)) for r in firsts]
    except Exception:
        continue
    total += 1
    shapes = {tuple(int(s) for s in im.shape[:3]) for im in imgs}
    same_shape += len(shapes) == 1
    a0 = imgs[0].affine
    same_affine += all(np.allclose(im.affine, a0, atol=1e-3) for im in imgs)

print(f"sampled {total} multi-contrast sessions")
print(f"  identical shape  across contrasts: {same_shape}/{total}")
print(f"  identical affine across contrasts: {same_affine}/{total}")
print("\nidentical affine => already co-registered onto a common grid (BraTS-like).")
print("shapes agree but affines do not => same matrix size, different scanner geometry.")
print("neither => registration + resampling is a required preprocessing step.")

## 7. Intensity distributions per contrast

Whether the volumes share an intensity scale decides how much normalization the pipeline owes
this data. `subplot_hists` handles the four things that make these comparable -- foreground
masking, a percentile range instead of the full tail, one set of bin edges per panel, and density
rather than count -- so this cell only has to choose the voxels.

In [ ]:
from visualization import subplot_hists, subplot_images

N_SUBJECTS     = 6           # sessions to overlay
HIST_CONTRASTS = [l for l in LABELS if l != "seg"]
MAX_VOXELS     = 200_000     # cap per volume, so 6 x 7 volumes stay in memory
BG_FRAC        = 0.05        # drop voxels below BG_FRAC * p99.5 -- air and the noise floor
BINS           = 120

_rng_h = np.random.default_rng(0)


def fg_samples(path, max_voxels=MAX_VOXELS, bg_frac=BG_FRAC):
    """Foreground voxel samples from one volume.

    NYUMets is not necessarily skull-stripped, so air is a low-but-nonzero noise floor rather
    than exact zeros -- `mask=` has nothing to key on and a plain `> 0` keeps all of it.
    Thresholding relative to the volume's own robust max drops it without a brain extractor.
    Set bg_frac=0 if the data turns out to be stripped after all.
    """
    v = np.asanyarray(nib.load(path).dataobj)
    while v.ndim > 3:                      # 4D (echoes / b-values) -> first volume
        v = v[..., 0]
    v = v.astype(np.float32).ravel()
    v = v[np.isfinite(v)]
    if v.size == 0:
        return v
    v = v[v > bg_frac * np.percentile(v, 99.5)]
    if v.size > max_voxels:
        v = v[_rng_h.choice(v.size, max_voxels, replace=False)]
    return v


# sessions with the most contrasts, so the same subjects appear in as many panels as possible
pool = full_sess if full_sess else sorted(cell, key=lambda k: -len(cell[k]))
keys = pool[:N_SUBJECTS]

samples = {lab: {} for lab in HIST_CONTRASTS}
for k in keys:
    for lab in HIST_CONTRASTS:
        if lab not in cell[k]:
            continue
        try:
            s = fg_samples(full_path(cell[k][lab][0]))
        except Exception as err:
            print(f"  !! {k} {lab}: {err}")
            continue
        if s.size:
            samples[lab][k] = s

present = [l for l in HIST_CONTRASTS if samples[l]]
print(f"{len(keys)} sessions, contrasts with data: {present}")

In [ ]:
# Top row raw, bottom row divided by each volume's own p99.5. Panels are NOT share_bins --
# contrasts live on different scales and should not be forced onto one axis; within a panel
# plot_hist already pools the series to one set of edges, which is the comparison that matters.
N = len(present)
lab_of = lambda k: f"{k[0]}/{k[1]}"

panels  = [{lab_of(k): v for k, v in samples[l].items()} for l in present]
panels += [{lab_of(k): v / np.percentile(v, 99.5) for k, v in samples[l].items()}
           for l in present]

# plot_hist passes color=None straight to ax.hist, and an EXPLICIT None disables matplotlib's
# property cycle -- every step-histogram then draws black and the legend is useless. So name the
# colours. As a TUPLE, not a list: subplot_hists spreads any *list* whose length matches the
# panel count across the panels, which would silently fire when len(keys) == 2*len(present).
_cyc = plt.rcParams["axes.prop_cycle"].by_key()["color"]
colors = tuple(_cyc[i % len(_cyc)] for i in range(len(keys)))

subplot_hists(
    panels,
    ncols=N,
    titles=[f"{l}  (n={len(samples[l])})" for l in present] + [f"{l} / p99.5" for l in present],
    bins=BINS,
    p=(0.5, 99.5),                                   # raw row: percentile range per panel
    range=[None] * N + [(0.0, 1.4)] * N,             # normalised row: one fixed range
    color=colors,
    xlabel=["raw intensity"] * N + ["/ per-volume p99.5"] * N,
    legend=[True] + [False] * (2 * N - 1),           # one legend is enough for the figure
    panel_size=(3.3, 3.0),
    suptitle="voxel intensity distribution per contrast, one line per session",
)

In [ ]:
# --- the number that decides whether per-subject normalisation is mandatory ---
print(f"{'contrast':<8} {'n':>3} {'med p50':>10} {'med p95':>10} {'med p99.5':>11} "
      f"{'p99.5 max/min':>14}")
for lab in present:
    q = np.array([np.percentile(v, [50, 95, 99.5]) for v in samples[lab].values()])
    spread = q[:, 2].max() / max(q[:, 2].min(), 1e-9)
    print(f"{lab:<8} {len(q):>3} {np.median(q[:,0]):>10.1f} {np.median(q[:,1]):>10.1f} "
          f"{np.median(q[:,2]):>11.1f} {spread:>14.2f}")

print("\np99.5 max/min ~1 => volumes already share a scale; >>1 => per-subject normalisation")
print("is required before any additive/percentile contrast model. If the bottom-row curves")
print("superimpose, dividing by p99.5 is sufficient; if they do not, the tail is being set by")
print("pathology rather than normal tissue and a lower percentile is needed.")
print("\nADC is a quantitative map in physical units, so its spread should be ~1.0 even when")
print("the weighted contrasts scatter. If it is not, suspect the ADC files, not the scaling.")

## 8. Eyeball one complete visit

In [ ]:
def mid_slice(path):
    """Middle axial slice of a volume, canonicalised to RAS so orientation is comparable."""
    img = nib.as_closest_canonical(nib.load(path))
    vol = np.asanyarray(img.dataobj)
    while vol.ndim > 3:
        vol = vol[..., 0]
    z = vol.shape[2] // 2
    return np.rot90(vol[:, :, z].astype(np.float32))


key = full_sess[0] if full_sess else max(cell, key=lambda k: len(cell[k]))
d = cell[key]
labs = sorted(d, key=LABELS.index)
print("showing", key, "->", labs)

imgs = [mid_slice(full_path(d[lab][0])) for lab in labs]

# share_window=False is the point: each contrast gets its own percentile stretch. The default
# shared window would scale every panel to whichever contrast has the largest dynamic range and
# black out the rest -- exactly what section 7's raw row shows happening.
subplot_images(
    imgs,
    titles=[f"{lab}  {im.shape}" for lab, im in zip(labs, imgs)],
    p=(1, 99),
    share_window=False,
    suptitle=f"{key[0]}   {key[1]}",
)

## 9. Tidy CSV

One row per `(patient, session, contrast, file)` with the path relative to `PATIENT_ROOT`.
This is what a `datasets/NYUMets/` registrar should read later, so it does not have to
re-derive any of the above.

In [ ]:
sess_order = {}
for p, ss in sess_by_pat.items():
    for i, s in enumerate(sorted(ss)):
        sess_order[(p, s)] = i

out = []
for (p, s), d in sorted(cell.items()):
    for lab in sorted(d, key=LABELS.index):
        for r in d[lab]:
            out.append({"patient": p, "session": s, "visit_idx": sess_order[(p, s)],
                        "contrast": lab, "n_at_cell": len(d[lab]),
                        "relpath": "/".join(x for x in (p, r["reldir"], r["fname"]) if x),
                        "nbytes": r["nbytes"]})

cols = ["patient", "session", "visit_idx", "contrast", "n_at_cell", "relpath", "nbytes"]
with open(TIDY_CSV, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=cols)
    w.writeheader(); w.writerows(out)

print(f"wrote {len(out)} rows -> {TIDY_CSV}")
print("root for relpath:", PATIENT_ROOT)
for r in out[:5]:
    print(" ", r)